In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pint
import pint_xarray

In [ ]:
# TEMPERATURE = 0
# TEMPERATURE = 10
# TEMPERATURE = 20
TEMPERATURE = 30
export_folder = f"./full_global_seapodym/"
export_folder = f"./pacific_seapodym/"

In [ ]:
# (Path(export_folder) / "data/").mkdir(exist_ok=False, parents=True)
# (Path(export_folder) / "output/").mkdir(exist_ok=False, parents=True)

In [ ]:
time = pd.date_range(
    "2000-01-01",
    "2010-12-31",
)

In [ ]:
data = xr.open_zarr("../data/pacific_lmtl_forcings.zarr/")[
    ["current_u", "current_v", "primary_production", "temperature"]
].load()
data

In [ ]:
data["primary_production"] = (
    data["primary_production"].pint.quantify().pint.to("mg/m**2/day").pint.dequantify()
)

In [ ]:
data = data.rename(
    {
        "x": "longitude",
        "y": "latitude",
        "z": "depth",
    }
)

# Convert coordinates to float32 for ADMB compatibility
data["latitude"] = data["latitude"].astype(np.float32)
data["longitude"] = data["longitude"].astype(np.float32)
data["depth"] = data["depth"].astype(np.float32)

# Change depth from 0 to 1 (ADMB uses 1-based indexing)
data["depth"] = data["depth"] + 1

data.latitude.attrs.update({"standard_name": "latitude"})
data.longitude.attrs.update({"standard_name": "longitude"})
data.depth.attrs.update({"standard_name": "depth"})
data.time.attrs.update({"standard_name": "time"})

In [ ]:
data.temperature.attrs = {"standard_name": "temperature"}
data.primary_production.attrs = {"standard_name": "net primary production"}
data.current_u.attrs = {"standard_name": "longitudinal current"}
data.current_v.attrs = {"standard_name": "latitudinal current"}

In [ ]:
data = data.astype(np.float32)
# data = data.sel(latitude=slice(-75, 75))
# data = data.sel(latitude=slice(0, 20), longitude=slice(-160, -140))

In [ ]:
data

## Create mask


In [ ]:
# Create mask with int32 dtype for ADMB compatibility
mask = data.temperature.mean(["time", "depth"], skipna=False).notnull().astype("int32")
mask.attrs = {"standard_name": "mask"}
mask.name = "mask"
mask.plot()

In [ ]:
mask.to_netcdf(f"{export_folder}/mask.nc")
xr.load_dataset(f"{export_folder}/mask.nc")


## To float 64


## Export


In [ ]:
for date in data.time.values:
    date_as_string = pd.to_datetime(str(date)).strftime("%Y%m%d")
    data.sel(time=[date])[
        ["current_u", "current_v", "temperature", "primary_production"]
    ].to_netcdf(f"{export_folder}/data/data_{date_as_string}.nc")

This shell commands can be run from seapodym-lmtl directory.

```bash
❯ mpirun -mca btl ^openib -np 6 ./build/bin/seapodym-lmtl -P -G D1N1 -V error /Users/ash/Documents/Workspaces/Data/phd/SEAPODYM_LMTL/2024-12-13_-_Constant_fields_no_transport/zpk.tmpl.xml
```

```bash
❯ mpirun -mca btl ^openib -np 6 ./build/bin/seapodym-lmtl -B -G D1N1 -V error /Users/ash/Documents/Workspaces/Data/phd/SEAPODYM_LMTL/2024-12-13_-_Constant_fields_no_transport/zpk.tmpl.xml
```
